# GitHub Triager RL Training
## Training Llama-3.2-3B to triage GitHub issues using GRPO + TRL + Unsloth

**Environment:** https://huggingface.co/spaces/Kavya011/github-triager-rl  
**GitHub:** https://github.com/KavyaTejani/Github-Triager

In [ ]:
!pip install unsloth trl  transformers datasets peft accelerate matplotlib nest_asyncio
!git clone https://github.com/KavyaTejani/Github-Triager.git
import sys
sys.path.append("/content/Github-Triager")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.

In [ ]:
ENVIRONMENT_URL = "https://kavya011-github-triager-rl.hf.space"
MODEL_NAME      = "unsloth/Llama-3.2-3B-Instruct"
MAX_STEPS       = 200      # increase to 500+ for real training
BATCH_SIZE      = 4
NUM_GENERATIONS = 4        # GRPO rollouts per prompt

In [ ]:
from client import GitHubTriagerClient

ENVIRONMENT_URL = "https://kavya011-github-triager-rl.hf.space"

def test_connection():
  with GitHubTriagerClient(base_url=ENVIRONMENT_URL) as env:
    result = env.reset(task_id="label_classification")
    print("✅ Successfully connected to the environment!")
    print("Observation keys:", list(result.keys()))
    return result

obs = test_connection()
print(obs)

<coroutine object test_connection at 0x785ec2513c40>


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Model loaded successfully.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model loaded successfully.


In [ ]:
def rollout_single(task_id: str = "label_classification"):
  with GitHubTriagerClient(base_url=ENVIRONMENT_URL) as env:
    obs = env.reset(task_id=task_id)
    prompt = f"""You are a GitHub issue triager.
Issue Title: {obs['issue']['title']}
Issue Body: {obs['issue']['body']}
Task: Classify this issue. Respond with valid JSON only.
Format: {{"label": "<bug|feature|documentation|question|enhancement>"}}"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response[len(prompt):]   # strip prompt from output

    result = env.step({"response": response})
    return prompt, response, float(result.get("reward", 0.0))

# Sanity check
try:
    prompt, response, reward = rollout_single()
    print(f"Reward: {reward:.3f}")
    print(f"Model response: {response}")
except Exception as e:
    print(f"Rollout failed (expected if not on GPU): {e}")

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

Rollout failed (expected if not on GPU): Client error '422 Unprocessable Entity' for url 'https://kavya011-github-triager-rl.hf.space/step?session_id=6cebcc77-bcc2-428b-9495-56b4901aaa2c'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422


In [ ]:
import json
import re
import requests

def get_env_reward(completion: str) -> float:
  try:
    match = re.search(r'\{.*\}', str(completion), re.DOTALL)
    if match:
      action_data = json.loads(match.group(0))
    else:
      return 0.01 # Model didn't output JSON
  except:
    return 0.01


  try:
    url = f"{ENVIRONMENT_URL}/grade/label_classification"
    response = requests.post(url, json={"action": action_data}, timeout=10)

    if response.status_code == 200:
      return float(response.json().get("score", 0.01))

  except:
    pass
  return 0.01

def compute_reward(prompts, completions, **kwargs):
  return [get_env_reward(c[0]["content"] if isinstance(c, list) else c) for c in completions]

In [ ]:
from datasets import Dataset

def build_dataset(n_samples: int = 100):
  samples = []
  with GitHubTriagerClient(base_url=ENVIRONMENT_URL) as env:
    for i in range(n_samples):
      obs = env.reset(task_id="label_classification")
      prompt = f"""You are a GitHub issue triager.
Issue Title: {obs['issue']['title']}
Issue Body: {obs['issue']['body']}
Classify this issue. Respond with JSON only.
Format: {{"label": "<bug|feature|documentation|question|enhancement>"}}"""
      samples.append({"prompt": prompt})
  return Dataset.from_list(samples)

dataset = build_dataset(100)
print(f"Dataset ready: {len(dataset)} samples")

Dataset ready: 100 samples


In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="./outputs/github-triager-grpo",
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=NUM_GENERATIONS,
    max_steps=MAX_STEPS,
    learning_rate=5e-6,
    logging_steps=10,
    save_steps=20,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[compute_reward],
    tokenizer=tokenizer,
)

trainer.train()
print("Training complete.")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 4 | Total steps = 200
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
Passing `generation_config` together with generation-related arguments=({'disable_compile', 'pad_token_id', 'cache_implementation'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: Futur

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / compute_reward / mean,rewards / compute_reward / std
10,0.060932,0.097125,0.153908,226.687500,116.500000,256.000000,0.787500,128.250001,116.500000,141.500000,0.000011,0.097125,0.173402
20,-0.019368,0.051125,0.070838,235.837500,137.800000,256.000000,0.837500,90.800000,61.000000,118.700000,0.000013,0.051125,0.082983
30,0.008397,0.086375,0.127624,228.537500,116.500000,256.000000,0.825000,80.283334,65.300000,97.000000,0.000026,0.086375,0.157610
40,0.002681,0.102750,0.127773,237.350000,111.800000,256.000000,0.875000,92.450000,86.200000,98.700000,0.000064,0.102750,0.166229
50,0.028328,0.064375,0.072673,223.387500,98.200000,256.000000,0.787500,83.100000,47.000000,117.000000,0.000078,0.064375,0.095366
60,-0.003297,0.111500,0.161577,237.462500,156.300000,256.000000,0.825000,114.133334,79.500000,143.400000,0.000083,0.111500,0.211225
70,-0.009985,0.072000,0.082556,218.575000,83.600000,256.000000,0.775000,85.883334,58.000000,107.800000,0.000079,0.072000,0.108826
80,0.037770,0.050000,0.060510,232.687500,139.900000,256.000000,0.837500,105.333334,88.700000,128.300000,0.000096,0.050000,0.080735
90,0.001832,0.079875,0.110693,239.512500,176.900000,256.000000,0.875000,47.866669,23.300000,75.200000,0.000118,0.079875,0.135402
100,0.062947,0.075375,0.086090,230.700000,111.900000,256.000000,0.837500,68.883334,60.700000,75.200000,0.000108,0.075375,0.117662


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs("results", exist_ok=True)

log_history = trainer.state.log_history
steps        = [x["step"]   for x in log_history if "loss"   in x]
losses       = [x["loss"]   for x in log_history if "loss"   in x]
r_steps      = [x["step"]   for x in log_history if "reward" in x]
rewards      = [x["reward"] for x in log_history if "reward" in x]

# Loss curve
plt.figure(figsize=(10, 4))
plt.plot(steps, losses, color="royalblue", linewidth=2)
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("GitHub Triager — Training Loss (GRPO)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/loss_curve.png", dpi=150)
plt.show()
print("Saved: results/loss_curve.png")

# Reward curve
plt.figure(figsize=(10, 4))
plt.plot(r_steps, rewards, color="seagreen", linewidth=2)
plt.xlabel("Training Step")
plt.ylabel("Average Reward")
plt.title("GitHub Triager — Reward During Training (GRPO)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/reward_curve.png", dpi=150)
plt.show()
print("Saved: results/reward_curve.png")

In [ ]:
import random
import matplotlib.pyplot as plt

def evaluate_model(n_episodes=20):
  total = 0.0
  for _ in range(n_episodes):
    prompt = dataset[random.randint(0, len(dataset)-1)]["prompt"]
    inputs  = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=64)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)[len(prompt):]

    total += get_env_reward(response)

  return total / n_episodes

try:
  trained_avg = evaluate_model(20)
  baseline    = 0.10

  plt.figure(figsize=(6, 5))
  plt.bar(["Baseline", "Trained"], [baseline, trained_avg], color=["#ff6b6b", "#51cf66"], edgecolor="black")
  plt.ylabel("Average Reward")
  plt.title("Before vs After GRPO Training")
  plt.savefig("results/before_after_comparison.png", dpi=150)
  plt.show()
  print("Saved: results/before_after_comparison.png")
except Exception as e:
  print(f"Evaluation failed: {e}")

In [ ]:
model.save_pretrained("outputs/github-triager-adapter")
tokenizer.save_pretrained("outputs/github-triager-adapter")
print("Adapter saved. Do NOT upcast 4-bit model and merge — use adapter directly.")

In [ ]:
def test_reasoning(title,body):
  custom_prompt = f"""Analyze the following Github issue step-by-step.
Identify the likely component and the severity.
Finally, provide the triage JSON.

Title: {title}
Body: {body}

Analysis:"""
  inputs = tokenizer(custom_prompt, return_tensors="pt").to("cuda")
  outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7)
  print(tokenizer.decode(outputs[0], skip_special_tokens=True))

test_reasoning(
    "AttributeError: 'list' object has no attribute 'keys' when loading gemma"
    "Loading gemma-4b raises a misleading error instead of a version requiremnet."
)
